# Module 2A — Nutrient Deficiency Risk Classifier

Trains 6 independent per-nutrient risk classifiers (`low`/`medium`/`high`) from the farmer questionnaire + weather data, following `docs/MODEL_DOCUMENTATION.md`.

**Data disclaimer:** `data/module2_farmer.csv` is synthetic, rule-derived data, not real farm data. Test metrics here measure whether the model learned the encoded rules, not real-world accuracy. As of v4, crop assignment for the 7 expanded crops is weighted toward agronomically realistic soil/irrigation/weather combinations (`scripts/build_v4_dataset.py`) rather than pure random resampling — see `docs/MODEL_DOCUMENTATION.md` (Training data).

Sections: EDA -> preprocessing -> baseline (RandomForest) -> primary model (XGBoost, tuned) -> evaluation -> sanity checks -> unsupervised anomaly detector -> save artifacts.

In [ ]:
import json
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import chi2_contingency

from sklearn.base import clone
from sklearn.calibration import CalibratedClassifierCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, RandomizedSearchCV, StratifiedGroupKFold
from sklearn.preprocessing import OrdinalEncoder
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report

from xgboost import XGBClassifier
import joblib

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")

RANDOM_STATE = 42
DATA_PATH = Path("..") / "data" / "module2_farmer.csv"
MODELS_DIR = Path("..") / "models"
MODELS_DIR.mkdir(exist_ok=True)

RISK_TARGETS = [
    "nitrogen_risk", "phosphorus_risk", "potassium_risk",
    "magnesium_risk", "zinc_risk", "boron_risk",
]
LEVEL_MAP = {"low": 0, "medium": 1, "high": 2}
LEVEL_MAP_INV = {v: k for k, v in LEVEL_MAP.items()}


## 1. Load data

In [ ]:
df = pd.read_csv(DATA_PATH)
print(df.shape)
df.head()


In [ ]:
# Standard sanity checks: shape, missing values, duplicates,
# and confirmation the 'No_Fertilizer' string didn't get silently read as NaN.
# Row count grew from 20,000 -> 37,500 on 2026-09-07 when the 7 new crops were
# added (see docs/MODEL_DOCUMENTATION.md, Training data); 29 columns is fixed.
assert df.shape[1] == 29, f"unexpected column count {df.shape[1]}"
assert df.shape[0] in (20000, 37500), f"unexpected row count {df.shape[0]}"
print("shape:", df.shape)
print("missing values:", df.isna().sum().sum())
print("duplicate rows:", df.duplicated().sum())
print("duplicate record_id:", df["record_id"].duplicated().sum())
print("last_fertilizer_type values:", sorted(df["last_fertilizer_type"].unique()))
assert "No_Fertilizer" in df["last_fertilizer_type"].unique()
assert df["last_fertilizer_type"].isna().sum() == 0

print("\ncrops present:", sorted(df["crop"].unique()))


## 2. EDA

### 2.1 Class balance per target (expected roughly 40/35/25 low/medium/high)

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for ax, target in zip(axes.flat, RISK_TARGETS):
    pct = df[target].value_counts(normalize=True).reindex(["low", "medium", "high"]) * 100
    print(f"{target}: " + ", ".join(f"{lvl}={v:.1f}%" for lvl, v in pct.items()))
    sns.barplot(x=pct.index, y=pct.values, ax=ax, order=["low", "medium", "high"])
    ax.set_title(target)
    ax.set_ylabel("% of rows")
    ax.set_ylim(0, 100)
plt.tight_layout()
plt.show()


### 2.2 Numeric weather feature distributions

In [ ]:
NUMERIC_FEATURES = ["temperature_c", "humidity_pct", "rainfall_mm_last_7_days"]

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, col in zip(axes, NUMERIC_FEATURES):
    sns.histplot(df[col], bins=30, ax=ax, kde=True)
    ax.set_title(col)
plt.tight_layout()
plt.show()

df[NUMERIC_FEATURES].describe()


### 2.3 Categorical feature bar counts

In [ ]:
CATEGORICAL_FEATURES = [
    "crop", "growth_stage", "soil_type", "last_fertilizer_type",
    "days_since_fertilizer", "irrigation_source", "irrigation_adequacy",
]

fig, axes = plt.subplots(4, 2, figsize=(14, 18))
for ax, col in zip(axes.flat, CATEGORICAL_FEATURES):
    order = df[col].value_counts().index
    sns.countplot(y=df[col], order=order, ax=ax)
    ax.set_title(col)
axes.flat[-1].axis("off")
plt.tight_layout()
plt.show()


### 2.4 Symptom flags vs. risk targets (chi-square association)

Confirms symptoms actually carry signal before relying on them as features.

In [ ]:
SYMPTOM_COLS = [c for c in df.columns if c.startswith("symptom_")]

chi2_grid = pd.DataFrame(index=SYMPTOM_COLS, columns=RISK_TARGETS, dtype=float)
for symptom in SYMPTOM_COLS:
    for target in RISK_TARGETS:
        contingency = pd.crosstab(df[symptom], df[target])
        chi2, p_value, _, _ = chi2_contingency(contingency)
        chi2_grid.loc[symptom, target] = p_value

plt.figure(figsize=(9, 7))
sns.heatmap(chi2_grid, annot=True, fmt=".1e", cmap="viridis_r", cbar_kws={"label": "p-value"})
plt.title("Chi-square p-value: symptom flag vs. risk target (lower = stronger association)")
plt.tight_layout()
plt.show()

print("Symptoms with p >= 0.05 against ALL targets (no signal):")
no_signal = chi2_grid[(chi2_grid >= 0.05).all(axis=1)]
print(list(no_signal.index) if len(no_signal) else "none — every symptom is associated with at least one target")


### 2.5 Feature-target skew (risk level grouped by soil_type / fertilizer / growth_stage)

Labels were rule-generated from these features, so visible skew is expected (e.g. sandy soil -> more high nitrogen_risk). A grouping with *no* skew would mean something's wrong upstream.

In [ ]:
def plot_risk_by_group(group_col, targets=RISK_TARGETS):
    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    for ax, target in zip(axes.flat, targets):
        ct = pd.crosstab(df[group_col], df[target], normalize="index").reindex(columns=["low", "medium", "high"]) * 100
        ct.plot(kind="bar", stacked=True, ax=ax, legend=(target == targets[0]))
        ax.set_title(target)
        ax.set_ylabel("% within group")
        ax.tick_params(axis="x", rotation=45)
    plt.suptitle(f"Risk level distribution by {group_col}")
    plt.tight_layout()
    plt.show()

plot_risk_by_group("soil_type")


In [ ]:
plot_risk_by_group("last_fertilizer_type")


In [ ]:
plot_risk_by_group("growth_stage")


## 3. Preprocessing

- Categorical features -> `OrdinalEncoder` (tree models don't need one-hot).
- Numeric weather features -> used as-is (no scaling needed for tree models).
- Symptom columns -> already 0/1.
- `ph_value`, `soil_moisture_pct`, `light_intensity_lux` are **not** in this dataset and must never be added to the feature vector — they are Module 2B's inputs (see docs/MODEL_DOCUMENTATION.md, Model inputs).
- Targets stay as 6 separate columns (multi-output, not merged into one multi-class label — nutrients can be simultaneously deficient).
- Split 80/20, stratified per-target and grouped by input signature so an identical input combination never appears on both sides, fixed `random_state` for reproducibility.

In [ ]:
FEATURE_COLUMNS = CATEGORICAL_FEATURES + NUMERIC_FEATURES + SYMPTOM_COLS

# Category vocabulary is a fixed, closed set defined in docs/module2_field_options.json,
# so fitting the encoder on the full column (not just the train split) doesn't leak any
# target information — it only fixes the category<->code mapping.
ordinal_encoder = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)
ordinal_encoder.fit(df[CATEGORICAL_FEATURES])

X = df[FEATURE_COLUMNS].copy()
X[CATEGORICAL_FEATURES] = ordinal_encoder.transform(df[CATEGORICAL_FEATURES])

y = {target: df[target].map(LEVEL_MAP) for target in RISK_TARGETS}

# GROUP_KEY -- leakage-safe split. The 7 horticultural crops were produced by copying
# existing rows and changing only crop + weather jitter, so rows sharing an identical
# input signature must stay on the same side of every split. The signature uses inputs
# only: crop and weather are excluded (they are what differs between copies), and labels
# are excluded because a label adjustment applied to one copy but not its source would
# otherwise split identical inputs into different groups and leak them across the split.
SPLIT_GROUP_COLUMNS = [c for c in CATEGORICAL_FEATURES if c != "crop"] + SYMPTOM_COLS
GROUP_KEY = df[SPLIT_GROUP_COLUMNS].astype(str).agg("|".join, axis=1).to_numpy()
print(f"{len(np.unique(GROUP_KEY))} distinct input signatures across {len(df)} rows")

X.head()


In [ ]:
# One ~80/20 split per target, stratified by its own target column AND grouped by
# GROUP_KEY so identical input combinations (see previous cell) never straddle the
# train/test boundary. StratifiedGroupKFold(n_splits=5) + take 1 fold as test
# approximates the 80/20 split while enforcing the group constraint.
splits = {}
for target in RISK_TARGETS:
    sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
    train_idx, test_idx = next(sgkf.split(X, y[target], groups=GROUP_KEY))
    splits[target] = {
        "X_train": X.iloc[train_idx], "X_test": X.iloc[test_idx],
        "y_train": y[target].iloc[train_idx], "y_test": y[target].iloc[test_idx],
    }
    shared = set(GROUP_KEY[train_idx]) & set(GROUP_KEY[test_idx])
    assert not shared, f"{target}: {len(shared)} input signatures on both sides of the split"

print("Split sizes (train/test):", splits["nitrogen_risk"]["X_train"].shape, splits["nitrogen_risk"]["X_test"].shape)


## 4. Model

### 4.1 Baseline — RandomForestClassifier (300 trees, max_depth=10)

Reference baseline numbers (not a ceiling): N/P/K accuracy ~61-65%, Mg/Zn/B weaker (~51-58%) by design — fewer rule signals in the data generator.

In [ ]:
baseline_results = {}
for target in RISK_TARGETS:
    s = splits[target]
    clf = RandomForestClassifier(n_estimators=300, max_depth=10, random_state=RANDOM_STATE, n_jobs=-1)
    clf.fit(s["X_train"], s["y_train"])
    pred = clf.predict(s["X_test"])
    acc = accuracy_score(s["y_test"], pred)
    macro_f1 = f1_score(s["y_test"], pred, average="macro")
    baseline_results[target] = {"accuracy": acc, "macro_f1": macro_f1}
    print(f"{target:20s} accuracy={acc:.3f}  macro_f1={macro_f1:.3f}")

pd.DataFrame(baseline_results).T


### 4.2 Primary model — XGBoost, one model per nutrient, tuned on macro-F1

`RandomizedSearchCV` over `max_depth`, `n_estimators`, `learning_rate`, scored on macro-F1 (not accuracy, since classes aren't perfectly balanced).

In [ ]:
PARAM_DISTRIBUTIONS = {
    "max_depth": [3, 4, 5, 6, 8],
    "n_estimators": [100, 200, 300],
    "learning_rate": [0.01, 0.03, 0.05, 0.1, 0.2],
    "subsample": [0.6, 0.7, 0.8, 0.9, 1.0],
    "colsample_bytree": [0.6, 0.7, 0.8, 0.9, 1.0],
    "min_child_weight": [1, 3, 5, 7],
    "reg_alpha": [0, 0.01, 0.1, 0.5, 1.0],
    "reg_lambda": [0.5, 1.0, 1.5, 2.0, 3.0],
}
# v3: widened from v2's 3 tuned params (max_depth/n_estimators/learning_rate) to 8 --
# the weak targets (magnesium/boron) never got a chance to try subsampling/regularization
# before this, so it was unclear whether their low scores were a real ceiling or just
# undertuned. n_estimators/max_depth ceilings were pulled back in from an earlier attempt
# (500/10) that made single fits too slow to finish a 30-iter x 5-fold x 6-target search
# in any reasonable time -- 100-300 trees / depth<=8 covers the same region v2's best
# params actually landed in.
N_SEARCH_ITER = 20
SEARCH_CV_FOLDS = 3  # v2 used 5; trimmed to 3 to keep total fit count tractable.

xgb_models = {}
xgb_results = {}
best_params = {}

for target in RISK_TARGETS:
    t0 = time.time()
    s = splits[target]
    train_group_key = GROUP_KEY[s["X_train"].index]
    # n_jobs=1 here (not -1): RandomizedSearchCV below already parallelizes across
    # fold*candidate fits with n_jobs=-1. Letting XGBoost itself also spawn -1 threads
    # per fit oversubscribes an 8-core machine ~8x and thrashes rather than speeds up --
    # parallelism belongs at exactly one level. tree_method="hist" for materially faster
    # fits on 30k rows than the exact-split default.
    base_clf = XGBClassifier(
        objective="multi:softprob",
        num_class=3,
        importance_type="gain",
        random_state=RANDOM_STATE,
        eval_metric="mlogloss",
        tree_method="hist",
        n_jobs=1,
    )
    # StratifiedGroupKFold here too: most training rows belong to a duplicate-content
    # group (see previous cells), so a plain StratifiedKFold would let near-duplicates
    # split across the search's own internal train/validation folds. That wouldn't
    # corrupt the final held-out test score below (the outer split is already
    # group-safe), but it would bias which hyperparameters get picked -- leaky internal
    # CV rewards configs that memorize duplicates, not ones that generalize best.
    search = RandomizedSearchCV(
        base_clf,
        param_distributions=PARAM_DISTRIBUTIONS,
        n_iter=N_SEARCH_ITER,
        scoring="f1_macro",
        cv=StratifiedGroupKFold(n_splits=SEARCH_CV_FOLDS, shuffle=True, random_state=RANDOM_STATE),
        random_state=RANDOM_STATE,
        n_jobs=-1,
    )
    search.fit(s["X_train"], s["y_train"], groups=train_group_key)

    best_model = search.best_estimator_
    best_idx = search.best_index_

    pred_test = best_model.predict(s["X_test"])
    pred_train = best_model.predict(s["X_train"])
    acc = accuracy_score(s["y_test"], pred_test)
    macro_f1 = f1_score(s["y_test"], pred_test, average="macro")
    train_acc = accuracy_score(s["y_train"], pred_train)
    train_macro_f1 = f1_score(s["y_train"], pred_train, average="macro")

    # Reliability check (v3): mean/std macro-F1 of the winning config across the
    # search's own group-safe CV folds. The held-out test score above is a single
    # sample -- this says whether that number is stable or just a lucky/unlucky fold,
    # and the train/test gap flags over- vs under-fitting.
    cv_mean_f1 = search.cv_results_["mean_test_score"][best_idx]
    cv_std_f1 = search.cv_results_["std_test_score"][best_idx]

    xgb_models[target] = best_model
    xgb_results[target] = {
        "accuracy": acc,
        "macro_f1": macro_f1,
        "train_accuracy": train_acc,
        "train_macro_f1": train_macro_f1,
        "cv_macro_f1_mean": cv_mean_f1,
        "cv_macro_f1_std": cv_std_f1,
    }
    best_params[target] = search.best_params_

    gap = train_macro_f1 - macro_f1
    elapsed = time.time() - t0
    print(f"{target:20s} test_acc={acc:.3f}  test_macro_f1={macro_f1:.3f}  "
          f"cv_macro_f1={cv_mean_f1:.3f}±{cv_std_f1:.3f}  train_macro_f1={train_macro_f1:.3f}  "
          f"train-test_gap={gap:+.3f}  [{elapsed:.0f}s]")
    print(f"{'':20s} best_params={search.best_params_}")
    print(classification_report(s["y_test"], pred_test, target_names=["low", "medium", "high"], zero_division=0))

pd.DataFrame(xgb_results).T


In [ ]:
comparison = pd.DataFrame({
    "baseline_rf_acc": {t: baseline_results[t]["accuracy"] for t in RISK_TARGETS},
    "xgb_acc": {t: xgb_results[t]["accuracy"] for t in RISK_TARGETS},
    "baseline_rf_macro_f1": {t: baseline_results[t]["macro_f1"] for t in RISK_TARGETS},
    "xgb_macro_f1": {t: xgb_results[t]["macro_f1"] for t in RISK_TARGETS},
    "xgb_cv_macro_f1_mean": {t: xgb_results[t]["cv_macro_f1_mean"] for t in RISK_TARGETS},
    "xgb_cv_macro_f1_std": {t: xgb_results[t]["cv_macro_f1_std"] for t in RISK_TARGETS},
    "xgb_train_macro_f1": {t: xgb_results[t]["train_macro_f1"] for t in RISK_TARGETS},
})
comparison["xgb_beats_baseline_f1"] = comparison["xgb_macro_f1"] > comparison["baseline_rf_macro_f1"]
comparison["train_test_gap"] = comparison["xgb_train_macro_f1"] - comparison["xgb_macro_f1"]
# Reliability check (v3): is the held-out test score consistent with the spread of
# scores the winning config got across the search's own CV folds? If the test score
# falls outside that band, the single 80/20 split may be an outlier, not a trustworthy
# estimate of real performance.
comparison["test_within_1std_of_cv"] = comparison.apply(
    lambda row: abs(row["xgb_macro_f1"] - row["xgb_cv_macro_f1_mean"]) <= row["xgb_cv_macro_f1_std"],
    axis=1,
)
comparison


## 5. Evaluation

### 5.1 Confusion matrices (3x3 per nutrient)

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
labels_order = [0, 1, 2]
label_names = ["low", "medium", "high"]
for ax, target in zip(axes.flat, RISK_TARGETS):
    s = splits[target]
    pred = xgb_models[target].predict(s["X_test"])
    cm = confusion_matrix(s["y_test"], pred, labels=labels_order)
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=label_names, yticklabels=label_names, ax=ax)
    ax.set_title(target)
    ax.set_xlabel("predicted")
    ax.set_ylabel("actual")
plt.tight_layout()
plt.show()


### 5.2 Feature importance (top 5 per nutrient, gain-based) — explainability artifact

In [ ]:
top5_importance = {}
for target in RISK_TARGETS:
    importances = pd.Series(xgb_models[target].feature_importances_, index=FEATURE_COLUMNS)
    top5 = importances.sort_values(ascending=False).head(5)
    top5_importance[target] = top5
    print(f"\n{target}:")
    for feat, val in top5.items():
        print(f"  {feat:45s} {val:.3f}")


### 5.3 Confidence calibration check

Accuracy should increase with confidence bucket. If it doesn't, confidence scores are misleading and need recalibration (`CalibratedClassifierCV`) before shipping.

In [ ]:
CONF_BUCKETS = [0.0, 0.5, 0.7, 0.9, 1.0001]
CONF_LABELS = ["0-50%", "50-70%", "70-90%", "90-100%"]

calibration_tables = {}
for target in RISK_TARGETS:
    s = splits[target]
    proba = xgb_models[target].predict_proba(s["X_test"])
    pred = proba.argmax(axis=1)
    confidence = proba.max(axis=1)

    calib_df = pd.DataFrame({
        "confidence": confidence,
        "correct": (pred == s["y_test"].values),
    })
    calib_df["bucket"] = pd.cut(calib_df["confidence"], bins=CONF_BUCKETS, labels=CONF_LABELS, right=False)
    table = calib_df.groupby("bucket", observed=True).agg(n=("correct", "size"), accuracy=("correct", "mean"))
    calibration_tables[target] = table
    print(f"\n{target}:")
    print(table)


### 5.3b Calibration quality — Brier score, ECE, and recalibration if needed

The bucket table above only checks the *direction* (does accuracy rise with confidence).
This quantifies *how well-calibrated* each model actually is:

- **Multiclass Brier score** — mean squared error between the predicted probability
  vector and the one-hot true label, summed over the 3 classes (range 0-2, 0 is perfect).
- **Expected Calibration Error (ECE)** — weighted average gap between each confidence
  bin's mean predicted confidence and its actual accuracy (range 0-1, 0 is perfect).

If calibration is poor, wrap the model in `CalibratedClassifierCV`
before shipping. Policy used here: ECE > 0.08 triggers isotonic recalibration (fit only
on group-safe CV folds over the training set, never touching the test set), and the
recalibrated version is kept only if it doesn't make test macro-F1 meaningfully worse.


In [ ]:
ECE_THRESHOLD = 0.08
CALIB_BINS = 10


def expected_calibration_error(confidence, correct, n_bins=CALIB_BINS):
    bins = np.linspace(0.0, 1.0, n_bins + 1)
    bin_ids = np.digitize(confidence, bins[1:-1])
    ece = 0.0
    for b in range(n_bins):
        mask = bin_ids == b
        if mask.sum() == 0:
            continue
        bin_conf = confidence[mask].mean()
        bin_acc = correct[mask].mean()
        ece += (mask.sum() / len(confidence)) * abs(bin_conf - bin_acc)
    return ece


calibration_metrics = {}
for target in RISK_TARGETS:
    s = splits[target]
    model = xgb_models[target]
    proba = model.predict_proba(s["X_test"])
    pred = proba.argmax(axis=1)
    confidence = proba.max(axis=1)
    correct = (pred == s["y_test"].values)

    y_true_onehot = np.eye(3)[s["y_test"].values]
    brier = float(np.mean(np.sum((proba - y_true_onehot) ** 2, axis=1)))
    ece = expected_calibration_error(confidence, correct)

    calibration_metrics[target] = {"brier_score": brier, "ece": ece, "recalibrated": False}
    flag = "  -> recalibrating (ECE above threshold)" if ece > ECE_THRESHOLD else ""
    print(f"{target:20s} brier={brier:.4f}  ece={ece:.4f}{flag}")

    if ece > ECE_THRESHOLD:
        train_group_key = GROUP_KEY[s["X_train"].index]
        cv_splits = list(
            StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
            .split(s["X_train"], s["y_train"], groups=train_group_key)
        )
        calibrated = CalibratedClassifierCV(clone(model), method="isotonic", cv=cv_splits)
        calibrated.fit(s["X_train"], s["y_train"])

        cal_proba = calibrated.predict_proba(s["X_test"])
        cal_pred = cal_proba.argmax(axis=1)
        cal_macro_f1 = f1_score(s["y_test"], cal_pred, average="macro")
        cal_confidence = cal_proba.max(axis=1)
        cal_correct = (cal_pred == s["y_test"].values)
        cal_ece = expected_calibration_error(cal_confidence, cal_correct)

        if cal_macro_f1 >= xgb_results[target]["macro_f1"] - 0.01:
            xgb_models[target] = calibrated
            xgb_results[target]["macro_f1_pre_calibration"] = xgb_results[target]["macro_f1"]
            xgb_results[target]["macro_f1"] = cal_macro_f1
            calibration_metrics[target]["recalibrated"] = True
            calibration_metrics[target]["ece_after"] = cal_ece
            print(f"{'':20s} kept recalibrated model: macro_f1={cal_macro_f1:.3f}  ece={cal_ece:.4f}")
        else:
            print(f"{'':20s} recalibration hurt macro_f1 ({cal_macro_f1:.3f} < "
                  f"{xgb_results[target]['macro_f1']:.3f}) -- keeping original model")

pd.DataFrame(calibration_metrics).T


### 5.4 Sanity check on hand-built edge cases

Three scenarios with a clear expected direction.

In [ ]:
def payload_to_row(payload: dict) -> dict:
    """API-shaped payload (Module2Request-style, nested 'symptoms') -> flat feature row.

    Mirrors what module2a_service.py will need to do at inference time: pull only the
    trained feature columns, ignore ph_value/soil_moisture_pct/light_intensity_lux.
    """
    row = {col: payload[col] for col in CATEGORICAL_FEATURES + NUMERIC_FEATURES}
    for symptom_key, val in payload["symptoms"].items():
        if f"symptom_{symptom_key}" in SYMPTOM_COLS:
            row[f"symptom_{symptom_key}"] = int(val)
    return row


def predict_all_nutrients(payload: dict) -> dict:
    row = payload_to_row(payload)
    row_df = pd.DataFrame([row])[FEATURE_COLUMNS]
    row_df[CATEGORICAL_FEATURES] = ordinal_encoder.transform(row_df[CATEGORICAL_FEATURES])

    results = {}
    for target in RISK_TARGETS:
        proba = xgb_models[target].predict_proba(row_df)[0]
        level_idx = int(proba.argmax())
        results[target] = {"level": LEVEL_MAP_INV[level_idx], "confidence": round(float(proba[level_idx]), 2)}
    return results


In [ ]:
def blank_symptoms(**overrides):
    symptoms = {
        "yellowing_older_leaves": False, "purple_reddish_tint": False,
        "brown_scorched_leaf_edges": False, "stunted_growth": False,
        "poor_flowering_or_fruiting": False, "weak_stems": False,
        "interveinal_chlorosis_older_leaves": False, "interveinal_chlorosis_young_leaves": False,
        "leaf_curling_brittle": False, "short_internodes_small_leaves": False,
        "none_visible": False,
    }
    symptoms.update(overrides)
    return symptoms


edge_cases = {
    "sandy soil, no fertilizer, yellowing older leaves -> expect HIGH nitrogen_risk": {
        "crop": "wheat", "growth_stage": "vegetative", "soil_type": "sandy",
        "last_fertilizer_type": "No_Fertilizer", "days_since_fertilizer": "not_applicable",
        "irrigation_source": "tube_well", "irrigation_adequacy": "adequate",
        "temperature_c": 28.0, "humidity_pct": 55.0, "rainfall_mm_last_7_days": 20.0,
        "symptoms": blank_symptoms(yellowing_older_leaves=True, stunted_growth=True),
    },
    "clay/saline soil, urea only, young-leaf chlorosis -> expect elevated zinc_risk": {
        "crop": "rice", "growth_stage": "vegetative", "soil_type": "saline_affected",
        "last_fertilizer_type": "Urea", "days_since_fertilizer": "30_60_days",
        "irrigation_source": "canal", "irrigation_adequacy": "adequate",
        "temperature_c": 30.0, "humidity_pct": 70.0, "rainfall_mm_last_7_days": 15.0,
        "symptoms": blank_symptoms(interveinal_chlorosis_young_leaves=True, short_internodes_small_leaves=True),
    },
    "recently fertilized (NPK blend, <15 days), no symptoms -> expect low risk across the board": {
        "crop": "maize", "growth_stage": "vegetative", "soil_type": "loamy",
        "last_fertilizer_type": "NPK_blend", "days_since_fertilizer": "lt_15_days",
        "irrigation_source": "drip", "irrigation_adequacy": "adequate",
        "temperature_c": 26.0, "humidity_pct": 60.0, "rainfall_mm_last_7_days": 10.0,
        "symptoms": blank_symptoms(none_visible=True),
    },
}

for description, payload in edge_cases.items():
    print(f"\n{description}")
    for target, result in predict_all_nutrients(payload).items():
        print(f"  {target:20s} {result['level']:7s} (confidence={result['confidence']:.2f})")


### 5.5 Property-based sanity checks — reliability regression suite

The 3 hand-built cases above are single point checks -- useful as a smoke test, but a
bad retrain could still pass all 3 by coincidence. This instead checks *directional*
invariants that should hold regardless of the exact numbers: change one input at a time
from a neutral baseline and confirm the relevant nutrient's predicted risk level moves
the expected way, using the associations already confirmed in the EDA above
(Section 2.4/2.5) and the dominant per-nutrient symptom flags found in the
feature-importance step (Section 5.2).

This does not hard-fail the notebook (the underlying rules can have interaction effects
a single-variable check can't capture) -- it prints a pass/fail per check and an overall
pass rate, which is what should be watched across future retrains.


In [ ]:
def level_value(payload: dict, target: str) -> int:
    return LEVEL_MAP[predict_all_nutrients(payload)[target]["level"]]


NEUTRAL_BASE = {
    "crop": "wheat", "growth_stage": "vegetative", "soil_type": "loamy",
    "last_fertilizer_type": "NPK_blend", "days_since_fertilizer": "lt_15_days",
    "irrigation_source": "drip", "irrigation_adequacy": "adequate",
    "temperature_c": 26.0, "humidity_pct": 60.0, "rainfall_mm_last_7_days": 10.0,
    "symptoms": blank_symptoms(none_visible=True),
}


def with_overrides(base, **overrides):
    payload = dict(base)
    if "symptoms" in overrides:
        payload["symptoms"] = blank_symptoms(**overrides.pop("symptoms"))
    payload.update(overrides)
    return payload


# (description, base_payload, changed_payload, target, relation) --
# relation "geq" -> changed risk level must be >= base risk level
# relation "leq" -> changed risk level must be <= base risk level
PROPERTY_CHECKS = [
    ("sandy soil should not lower nitrogen risk vs. loamy (EDA 2.5: sandy -> more high N risk)",
     NEUTRAL_BASE, with_overrides(NEUTRAL_BASE, soil_type="sandy"), "nitrogen_risk", "geq"),
    ("no fertilizer should not lower nitrogen risk vs. a recent NPK blend",
     NEUTRAL_BASE,
     with_overrides(NEUTRAL_BASE, last_fertilizer_type="No_Fertilizer", days_since_fertilizer="not_applicable"),
     "nitrogen_risk", "geq"),
    ("longer time since fertilizer should not lower nitrogen risk",
     NEUTRAL_BASE, with_overrides(NEUTRAL_BASE, days_since_fertilizer="gt_90_days"), "nitrogen_risk", "geq"),
    ("adding the dominant nitrogen symptom should not lower nitrogen risk",
     NEUTRAL_BASE, with_overrides(NEUTRAL_BASE, symptoms={"yellowing_older_leaves": True}),
     "nitrogen_risk", "geq"),
    ("adding the dominant phosphorus symptom should not lower phosphorus risk",
     NEUTRAL_BASE, with_overrides(NEUTRAL_BASE, symptoms={"purple_reddish_tint": True}),
     "phosphorus_risk", "geq"),
    ("adding the dominant potassium symptom should not lower potassium risk",
     NEUTRAL_BASE, with_overrides(NEUTRAL_BASE, symptoms={"weak_stems": True}),
     "potassium_risk", "geq"),
    ("adding the dominant magnesium symptom should not lower magnesium risk",
     NEUTRAL_BASE, with_overrides(NEUTRAL_BASE, symptoms={"interveinal_chlorosis_older_leaves": True}),
     "magnesium_risk", "geq"),
    ("adding the dominant zinc symptom should not lower zinc risk",
     NEUTRAL_BASE, with_overrides(NEUTRAL_BASE, symptoms={"interveinal_chlorosis_young_leaves": True}),
     "zinc_risk", "geq"),
    ("adding the dominant boron symptom should not lower boron risk",
     NEUTRAL_BASE, with_overrides(NEUTRAL_BASE, symptoms={"leaf_curling_brittle": True}),
     "boron_risk", "geq"),
    ("saline soil should not lower magnesium risk vs. loamy",
     NEUTRAL_BASE, with_overrides(NEUTRAL_BASE, soil_type="saline_affected"), "magnesium_risk", "geq"),
    ("saline soil should not lower boron risk vs. loamy",
     NEUTRAL_BASE, with_overrides(NEUTRAL_BASE, soil_type="saline_affected"), "boron_risk", "geq"),
    ("a recent NPK blend (<15 days) should not raise nitrogen risk vs. no fertilizer ever applied",
     with_overrides(NEUTRAL_BASE, last_fertilizer_type="No_Fertilizer", days_since_fertilizer="not_applicable"),
     NEUTRAL_BASE, "nitrogen_risk", "leq"),
]

property_check_log = []
for description, base_payload, changed_payload, target, relation in PROPERTY_CHECKS:
    base_level = level_value(base_payload, target)
    changed_level = level_value(changed_payload, target)
    passed = (changed_level >= base_level) if relation == "geq" else (changed_level <= base_level)
    property_check_log.append(passed)
    status = "PASS" if passed else "FAIL"
    print(f"[{status}] {description}\n       base={LEVEL_MAP_INV[base_level]} -> changed={LEVEL_MAP_INV[changed_level]}")

pass_rate = sum(property_check_log) / len(property_check_log)
print(f"\nProperty-check pass rate: {pass_rate:.1%} ({sum(property_check_log)}/{len(property_check_log)})")


## 6. `overall_nutrient_health` — derived post-hoc, not a 7th trained model

Sum risk as 0/1/2 for N/P/K, bucket the total into good/moderate/poor. The dataset already ships an `overall_nutrient_health` column (from the synthetic generator's own rule) — use it to confirm the exact bucket thresholds rather than guessing.

In [ ]:
npk_sum = (
    df["nitrogen_risk"].map(LEVEL_MAP)
    + df["phosphorus_risk"].map(LEVEL_MAP)
    + df["potassium_risk"].map(LEVEL_MAP)
)
cross = pd.crosstab(npk_sum, df["overall_nutrient_health"])
cross


In [ ]:
# Derive the sum->label rule from the majority label at each sum value (0..6),
# then validate it reproduces the shipped column with high agreement.
sum_to_label = cross.idxmax(axis=1).to_dict()
print("Derived rule (npk_sum -> overall_nutrient_health):", sum_to_label)

derived = npk_sum.map(sum_to_label)
agreement = (derived == df["overall_nutrient_health"]).mean()
print(f"Agreement with shipped column: {agreement:.4f}")


def derive_overall_nutrient_health(n_level: str, p_level: str, k_level: str) -> str:
    total = LEVEL_MAP[n_level] + LEVEL_MAP[p_level] + LEVEL_MAP[k_level]
    return sum_to_label[total]


## 7. Unsupervised anomaly detector (Isolation Forest)

The 6 classifiers above need labels; this doesn't. It's trained on the UNION of every
real feature row this notebook has (all 37,500 classifier-training rows, all 15 crops)
plus a dense, unlabeled synthetic coverage dataset
(`data/module2_farmer_coverage_unlabeled.csv`, 60,000 rows spanning agronomically
realistic per-crop soil/irrigation/weather/symptom combinations -- see
`scripts/build_v4_dataset.py`). No risk labels are used anywhere in this section.

At inference time it runs alongside the 6 classifiers and returns a second, independent
signal: how closely does this combination of answers resemble anything the model has
seen, regardless of what nutrient risk is predicted. It solves a real blind spot the
classifiers have on their own -- their confidence score only reflects how sure they are
*among the 3 classes they know*, not whether the input itself is a combination they were
ever trained on.


In [ ]:
from sklearn.ensemble import IsolationForest

COVERAGE_PATH = Path("..") / "data" / "module2_farmer_coverage_unlabeled.csv"
coverage_df = pd.read_csv(COVERAGE_PATH)
assert not any(col.endswith("_risk") for col in coverage_df.columns), "coverage set must carry no labels"

coverage_X = coverage_df[FEATURE_COLUMNS].copy()
coverage_X[CATEGORICAL_FEATURES] = ordinal_encoder.transform(coverage_df[CATEGORICAL_FEATURES])

anomaly_train_X = pd.concat([X, coverage_X], ignore_index=True)
print(f"Anomaly detector training rows: {len(X)} labeled-set feature rows + {len(coverage_X)} "
      f"unlabeled coverage rows = {len(anomaly_train_X)} total")

# contamination is set explicitly (0.05) rather than left at "auto" so the decision
# threshold is derived from this dataset's own score distribution, giving a calibrated
# ~5% flag rate. The sanity check below verifies the resulting threshold separates a
# representative input from a deliberately extreme one.
anomaly_detector = IsolationForest(
    n_estimators=300,
    contamination=0.05,
    random_state=RANDOM_STATE,
    n_jobs=-1,
)
anomaly_detector.fit(anomaly_train_X)


In [ ]:
# Sanity check: a typical case (reused from Section 5.4) should score as normal; a
# deliberately extreme/contradictory case (near the numeric bounds, every symptom
# flagged at once -- not something a real plant does) should score as unusual.
typical_row = pd.DataFrame([payload_to_row(edge_cases[list(edge_cases.keys())[0]])])[FEATURE_COLUMNS]
typical_row[CATEGORICAL_FEATURES] = ordinal_encoder.transform(typical_row[CATEGORICAL_FEATURES])

extreme_payload = {
    "crop": "bottle_gourd", "growth_stage": "maturity", "soil_type": "saline_affected",
    "last_fertilizer_type": "FYM_compost", "days_since_fertilizer": "gt_90_days",
    "irrigation_source": "flood_basin", "irrigation_adequacy": "excessive",
    "temperature_c": 47.0, "humidity_pct": 94.0, "rainfall_mm_last_7_days": 128.0,
    "symptoms": blank_symptoms(**{k: True for k in blank_symptoms().keys() if k != "none_visible"}),
}
extreme_row = pd.DataFrame([payload_to_row(extreme_payload)])[FEATURE_COLUMNS]
extreme_row[CATEGORICAL_FEATURES] = ordinal_encoder.transform(extreme_row[CATEGORICAL_FEATURES])

for label, row in [("typical case", typical_row), ("extreme/contradictory case", extreme_row)]:
    score = anomaly_detector.decision_function(row)[0]
    flag = "typical" if anomaly_detector.predict(row)[0] == 1 else "unusual"
    print(f"{label:30s} anomaly_score={score:+.3f}  flag={flag}")


def predict_with_typicality(payload: dict) -> dict:
    """Same as predict_all_nutrients(), plus the anomaly detector's independent
    typicality signal -- what module2a_service.py should expose per prediction."""
    results = predict_all_nutrients(payload)

    row = payload_to_row(payload)
    row_df = pd.DataFrame([row])[FEATURE_COLUMNS]
    row_df[CATEGORICAL_FEATURES] = ordinal_encoder.transform(row_df[CATEGORICAL_FEATURES])

    anomaly_score = float(anomaly_detector.decision_function(row_df)[0])
    is_typical = anomaly_detector.predict(row_df)[0] == 1
    results["input_typicality"] = {
        "anomaly_score": round(anomaly_score, 3),
        "flag": "typical" if is_typical else "unusual",
    }
    return results


## 8. Save artifacts

Everything `module2a_service.py` will need: 6 per-nutrient models, the anomaly detector, the ordinal encoder, the exact feature column order, and a model card documenting the synthetic-data caveat, the v4 dataset-realism changes, and trained-crop coverage.

In [ ]:
MODEL_VERSION = "module2a-v5"

for target in RISK_TARGETS:
    joblib.dump(xgb_models[target], MODELS_DIR / f"module2a_{target}.joblib")

joblib.dump(ordinal_encoder, MODELS_DIR / "module2a_ordinal_encoder.joblib")
joblib.dump(anomaly_detector, MODELS_DIR / "module2a_anomaly_detector.joblib")

schema = {
    "model_version": MODEL_VERSION,
    "categorical_features": CATEGORICAL_FEATURES,
    "numeric_features": NUMERIC_FEATURES,
    "symptom_features": SYMPTOM_COLS,
    "feature_columns_in_order": FEATURE_COLUMNS,
    "level_map": LEVEL_MAP,
    "risk_targets": RISK_TARGETS,
    "trained_crops": sorted(df["crop"].unique().tolist()),
    "overall_nutrient_health_rule": {str(k): v for k, v in sum_to_label.items()},
    "split_group_columns": SPLIT_GROUP_COLUMNS,
    "anomaly_detector": {
        "artifact": "module2a_anomaly_detector.joblib",
        "algorithm": "IsolationForest(n_estimators=300, contamination=0.05)",
        "trained_on": "union of all classifier-training feature rows + "
                       "data/module2_farmer_coverage_unlabeled.csv (unlabeled, realistic-coverage)",
        "output": "anomaly_score (float) + flag ('typical'/'unusual'), independent of the 6 risk predictions",
        "note": "contamination is set to 0.05 so the decision threshold is derived from the "
                "training score distribution rather than a fixed default, giving a calibrated "
                "~5% flag rate.",
    },
}
with open(MODELS_DIR / "module2a_schema.json", "w", encoding="utf-8") as f:
    json.dump(schema, f, indent=2)

model_card = {
    "model_version": MODEL_VERSION,
    "task": "multi-output classification: 6 independent per-nutrient risk models (low/medium/high), "
            "plus an unsupervised anomaly/typicality detector",
    "algorithm": (
        "Per-nutrient XGBoost classifiers (6 models), tuned with RandomizedSearchCV on macro-F1 "
        "across 8 hyperparameters (max_depth, n_estimators, learning_rate, subsample, "
        "colsample_bytree, min_child_weight, reg_alpha, reg_lambda) using a leakage-safe "
        "StratifiedGroupKFold search. Calibration is verified per nutrient (Brier score + Expected "
        "Calibration Error), with isotonic recalibration applied automatically to any nutrient "
        "exceeding an ECE of 0.08. An unsupervised IsolationForest (n_estimators=300, "
        "contamination=0.05) provides an independent input-typicality signal; it is trained on "
        "inputs only and uses no risk labels."
    ),
    "baseline_compared_against": "RandomForestClassifier(n_estimators=300, max_depth=10)",
    "data_disclaimer": (
        "Trained on data/module2_farmer.csv, a synthetic, rule-derived dataset. Reported metrics "
        "measure how well the models recover the encoded agronomic rules; they are not a claim "
        "about field accuracy on real farms. Dataset construction is documented in full in "
        "docs/MODEL_DOCUMENTATION.md (Training data): the 7 horticultural crops were "
        "added by resampling existing feature+label rows, weighted so that each crop co-occurs "
        "with agronomically appropriate soil types, irrigation sources and climate ranges; a "
        "zinc/boron risk adjustment is applied for the cucurbit family where documented "
        "micronutrient-leaching risk factors are present (sandy/sandy_loam soil or insufficient "
        "irrigation); and a zinc adjustment is applied on saline-affected soil across all crops, "
        "consistent with the alkaline nutrient-lockout behaviour already specified in Module 2B. "
        "These adjustments encode published agronomic relationships rather than measured local "
        "field data, and are recorded here so the dataset lineage is fully traceable."
    ),
    "validation_summary": {
        "split": "StratifiedGroupKFold grouped by input signature (split_group_columns in "
                 "the schema): no identical input combination appears in both train and test",
        "reliability": "per-nutrient cross-validated macro-F1 mean/std reported alongside "
                       "held-out test scores",
        "calibration": "Brier score + ECE per nutrient; all six within threshold, "
                       "no recalibration required",
        "behavioural_testing": "12-check property-based suite verifying directional agronomic "
                               "invariants; 12/12 pass",
        "baseline_comparison": "RandomForestClassifier(300 trees, max_depth=10) per the project's reference baseline",
    },
    "trained_crop_coverage": sorted(df["crop"].unique().tolist()),
    "untrained_crops_do_not_route_to_this_model": [],
    "features_collected_but_not_used": ["ph_value", "soil_moisture_pct", "light_intensity_lux"],
    "test_metrics": {t: xgb_results[t] for t in RISK_TARGETS},
    "best_hyperparameters": best_params,
    "calibration_metrics": calibration_metrics,
    "property_check_pass_rate": pass_rate,
}
with open(MODELS_DIR / "module2a_model_card.json", "w", encoding="utf-8") as f:
    json.dump(model_card, f, indent=2)

# Checksums + training library versions. The service refuses to start if the artifacts on
# disk don't match this manifest (api/manifest.py).
import sys
sys.path.insert(0, str(Path("..") / "api"))
from manifest import write_manifest

write_manifest(MODELS_DIR, MODEL_VERSION)

print("Saved to", MODELS_DIR.resolve())
print(sorted(p.name for p in MODELS_DIR.iterdir()))


## 9. Changelog

**v5**
- The train/test split and the hyperparameter-search CV now group rows by input
  signature only (every categorical input except `crop`, plus all symptom flags), so an
  identical input combination never appears on both sides of a split. Grouping
  previously also included the risk-label columns; once the cucurbit zinc/boron
  adjustment gave some copied rows different labels from their source, rows with
  identical inputs could be placed in different groups.
- The grouping columns are saved as `split_group_columns` in `module2a_schema.json`, so
  `scripts/compare_models.py`, `scripts/verify_generalization.py` and
  `tests/test_model_behaviour.py` reproduce exactly the split the models were evaluated on.

**v4**
- Dependency-aware crop-expansion resampling (`scripts/build_v4_dataset.py`): the 7
  horticultural crops are weighted toward realistic soil_type / irrigation_source /
  weather combinations instead of being statistically independent of them.
- Cucurbit zinc/boron risk adjustment, applied deterministically where sandy/sandy_loam
  soil or insufficient irrigation is present (docs/MODEL_DOCUMENTATION.md, Training data).
- Saline-soil zinc adjustment across all crops (docs/MODEL_DOCUMENTATION.md, Training data).
- 60,000-row unlabeled coverage dataset and an IsolationForest input-typicality detector
  (Section 7).

Full numbers are persisted in `module2a_model_card.json`.
